In [1]:
!pip install pyspark

In [3]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106],
    "supplier_id": [1, 2, 1, 3, 2, 1],
    "order_date": [
        "2025-01-01",
        "2025-01-03",
        "2025-01-06",
        "2025-01-08",
        "2025-01-10",
        "2025-01-12"
    ],
    "expected_delivery_date": [
        "2025-01-04",
        "2025-01-06",
        "2025-01-06",
        "2025-01-12",
        "2025-01-10",
        "2025-01-15"
    ],
    "delivery_date": [
        "2025-01-05",
        "2025-01-08",
        "2025-01-06",
        "2025-01-15",
        "2025-01-11",
        "2025-01-18"
    ]
})

orders.to_csv("orders.csv", index=False)

print("orders.csv created successfully")

orders.csv created successfully


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("SupplyChainMonitoring") \
    .getOrCreate()

print("Spark Started")

Spark Started


In [5]:
df = spark.read.csv(
    "orders.csv",
    header=True,
    inferSchema=True
)

print("Original Data")
df.show()

Original Data
+--------+-----------+----------+----------------------+-------------+
|order_id|supplier_id|order_date|expected_delivery_date|delivery_date|
+--------+-----------+----------+----------------------+-------------+
|     101|          1|2025-01-01|            2025-01-04|   2025-01-05|
|     102|          2|2025-01-03|            2025-01-06|   2025-01-08|
|     103|          1|2025-01-06|            2025-01-06|   2025-01-06|
|     104|          3|2025-01-08|            2025-01-12|   2025-01-15|
|     105|          2|2025-01-10|            2025-01-10|   2025-01-11|
|     106|          1|2025-01-12|            2025-01-15|   2025-01-18|
+--------+-----------+----------+----------------------+-------------+



In [6]:
df = df.withColumn(
    "delivery_date",
    to_date(col("delivery_date"))
)

df = df.withColumn(
    "expected_delivery_date",
    to_date(col("expected_delivery_date"))
)

In [7]:
df = df.withColumn(
    "delay_days",
    datediff(
        col("delivery_date"),
        col("expected_delivery_date")
    )
)

print("Data with Delay Calculation")
df.show()

Data with Delay Calculation
+--------+-----------+----------+----------------------+-------------+----------+
|order_id|supplier_id|order_date|expected_delivery_date|delivery_date|delay_days|
+--------+-----------+----------+----------------------+-------------+----------+
|     101|          1|2025-01-01|            2025-01-04|   2025-01-05|         1|
|     102|          2|2025-01-03|            2025-01-06|   2025-01-08|         2|
|     103|          1|2025-01-06|            2025-01-06|   2025-01-06|         0|
|     104|          3|2025-01-08|            2025-01-12|   2025-01-15|         3|
|     105|          2|2025-01-10|            2025-01-10|   2025-01-11|         1|
|     106|          1|2025-01-12|            2025-01-15|   2025-01-18|         3|
+--------+-----------+----------+----------------------+-------------+----------+



In [8]:
delayed_df = df.filter(
    col("delay_days") > 0
)

print("Delayed Shipments")
delayed_df.show()

Delayed Shipments
+--------+-----------+----------+----------------------+-------------+----------+
|order_id|supplier_id|order_date|expected_delivery_date|delivery_date|delay_days|
+--------+-----------+----------+----------------------+-------------+----------+
|     101|          1|2025-01-01|            2025-01-04|   2025-01-05|         1|
|     102|          2|2025-01-03|            2025-01-06|   2025-01-08|         2|
|     104|          3|2025-01-08|            2025-01-12|   2025-01-15|         3|
|     105|          2|2025-01-10|            2025-01-10|   2025-01-11|         1|
|     106|          1|2025-01-12|            2025-01-15|   2025-01-18|         3|
+--------+-----------+----------+----------------------+-------------+----------+



In [9]:
supplier_summary = delayed_df.groupBy(
    "supplier_id"
).agg(
    count("order_id").alias("delayed_orders")
)

print("Delayed Orders by Supplier")
supplier_summary.show()

Delayed Orders by Supplier
+-----------+--------------+
|supplier_id|delayed_orders|
+-----------+--------------+
|          1|             2|
|          3|             1|
|          2|             2|
+-----------+--------------+



In [10]:
supplier_summary.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/delayed_orders_csv")

In [11]:
supplier_summary.write \
    .mode("overwrite") \
    .parquet("/content/delayed_orders_parquet")

In [12]:
!zip -r delayed_orders_csv.zip delayed_orders_csv
!zip -r delayed_orders_parquet.zip delayed_orders_parquet

  adding: delayed_orders_csv/ (stored 0%)
  adding: delayed_orders_csv/._SUCCESS.crc (stored 0%)
  adding: delayed_orders_csv/.part-00000-3c218f33-d8ff-453a-885c-7ae9032d91ff-c000.csv.crc (stored 0%)
  adding: delayed_orders_csv/part-00000-3c218f33-d8ff-453a-885c-7ae9032d91ff-c000.csv (stored 0%)
  adding: delayed_orders_csv/_SUCCESS (stored 0%)
  adding: delayed_orders_parquet/ (stored 0%)
  adding: delayed_orders_parquet/._SUCCESS.crc (stored 0%)
  adding: delayed_orders_parquet/part-00000-9a4849bc-dc5d-49d6-9f39-e69da161a1c3-c000.snappy.parquet (deflated 35%)
  adding: delayed_orders_parquet/_SUCCESS (stored 0%)
  adding: delayed_orders_parquet/.part-00000-9a4849bc-dc5d-49d6-9f39-e69da161a1c3-c000.snappy.parquet.crc (stored 0%)


In [13]:
from google.colab import files

files.download("delayed_orders_csv.zip")
files.download("delayed_orders_parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>